# Air Quality Data in India — Analysis

**Reference:** [Kaggle: Air Quality Data in India](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india)

This notebook explores air-quality records using Pandas, NumPy, Matplotlib, and Seaborn.

> **Important:** Place the official Kaggle `city_day.csv` beside this notebook for your final submission. If it is missing, the notebook loads `sample_air_quality_data.csv`, which is only an illustrative sample and is not suitable for real findings.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
# Prefer the official Kaggle dataset. Fall back to the included illustrative sample only if needed.
if os.path.exists("city_day.csv"):
    df = pd.read_csv("city_day.csv")
    print("Loaded official dataset: city_day.csv")
elif os.path.exists("sample_air_quality_data.csv"):
    df = pd.read_csv("sample_air_quality_data.csv")
    warnings.warn("Using illustrative sample data. Download the official Kaggle city_day.csv for valid project findings.")
else:
    raise FileNotFoundError("Place city_day.csv or sample_air_quality_data.csv in the notebook folder.")

print("Dataset shape:", df.shape)
display(df.head())


## 1. Inspect and clean the data

In [ ]:
print("Column names:")
print(df.columns.tolist())
print("\\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))

# Standardize the date column when present; keep invalid dates as NaT.
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Convert expected measurement columns to numeric if present.
measurement_cols = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene", "AQI"]
for col in measurement_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Duplicate rows:", df.duplicated().sum())


## 2. Numeric summary

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
display(df[numeric_cols].describe().T if len(numeric_cols) else pd.DataFrame({"Info": ["No numeric columns found"]}))


## 3. AQI distribution

In [ ]:
if "AQI" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df["AQI"].dropna(), bins=20, kde=True)
    plt.title("Distribution of AQI")
    plt.xlabel("AQI")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.show()
else:
    print("AQI column not found.")


## 4. Cities with the highest average AQI

In [ ]:
if {"City", "AQI"}.issubset(df.columns):
    city_aqi = df.groupby("City", dropna=True)["AQI"].mean().dropna().sort_values(ascending=False).head(10)
    display(city_aqi.to_frame("Average AQI"))
    if not city_aqi.empty:
        plt.figure(figsize=(10, 5))
        city_aqi.sort_values().plot(kind="barh")
        plt.title("Top 10 Cities by Average AQI")
        plt.xlabel("Average AQI")
        plt.ylabel("City")
        plt.tight_layout()
        plt.show()
else:
    print("City or AQI column not found.")


## 5. Monthly AQI pattern

In [ ]:
if {"Date", "AQI"}.issubset(df.columns):
    dated = df.dropna(subset=["Date", "AQI"]).copy()
    if not dated.empty:
        dated["Month"] = dated["Date"].dt.month
        monthly_aqi = dated.groupby("Month")["AQI"].mean()
        display(monthly_aqi.to_frame("Average AQI"))
        monthly_aqi.plot(marker="o", figsize=(8, 4))
        plt.title("Average AQI by Month")
        plt.xlabel("Month")
        plt.ylabel("Average AQI")
        plt.xticks(range(1, 13))
        plt.tight_layout()
        plt.show()
else:
    print("Date or AQI column not found.")


## 6. Pollutant correlation

In [ ]:
available_pollutants = [c for c in measurement_cols if c in df.columns and c != "AQI"]
if len(available_pollutants) >= 2:
    corr = df[available_pollutants].corr(numeric_only=True)
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", center=0)
    plt.title("Correlation Between Pollutant Measurements")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough pollutant columns available for correlation analysis.")


## 7. AQI category counts

In [ ]:
bucket_col = "AQI_Bucket" if "AQI_Bucket" in df.columns else ("AQI_Bucket " if "AQI_Bucket " in df.columns else None)
if bucket_col:
    counts = df[bucket_col].value_counts(dropna=False)
    display(counts.to_frame("Record count"))
    counts.plot(kind="bar", figsize=(9, 4))
    plt.title("AQI Category Distribution")
    plt.xlabel("AQI category")
    plt.ylabel("Number of records")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("AQI category column not found.")


## 8. Compare 2019 and 2020 (if available)

In [ ]:
if {"Date", "AQI"}.issubset(df.columns):
    yearly = df.dropna(subset=["Date", "AQI"]).copy()
    yearly["Year"] = yearly["Date"].dt.year
    comparison = yearly[yearly["Year"].isin([2019, 2020])].groupby("Year")["AQI"].mean()
    if set(comparison.index) == {2019, 2020}:
        display(comparison.to_frame("Average AQI"))
        comparison.plot(kind="bar", figsize=(6, 4))
        plt.title("Average AQI: 2019 vs 2020")
        plt.xlabel("Year")
        plt.ylabel("Average AQI")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print("Both 2019 and 2020 are not present in the current data; comparison skipped.")
else:
    print("Date or AQI column not found.")


## Conclusion

Use the tables and plots generated from the **official Kaggle dataset** to write your final conclusions. Describe which cities or months have higher average AQI, which pollutant measurements appear correlated, and how AQI categories are distributed. Avoid writing specific findings until you have run the notebook with the real `city_day.csv` file.